In [1]:
import json
from collections import Counter
from pathlib import Path

METADATA_FILE = "../data2/processed/meta_filtered.jsonl"

# Your current generated dataset
INPUT_GENERATED_FILE = "../data2/generated_queries.jsonl"

# New cleaned dataset
CLEANED_FILE = "../data2/output.jsonl"

In [2]:
asin_to_category = {}

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        product = json.loads(line)

        asin = product.get("parent_asin")
        category = product.get("main_category")

        if asin and category:
            asin_to_category[asin] = category


print("Products in category mapping:", len(asin_to_category))

Products in category mapping: 549084


In [3]:
category_counts = Counter()
missing_category = 0
total_records = 0

with open(INPUT_GENERATED_FILE, "r", encoding="utf-8") as fin, \
     open(CLEANED_FILE, "w", encoding="utf-8") as fout:

    for line in fin:

        if not line.strip():
            continue

        record = json.loads(line)

        total_records += 1

        # Get target ASIN
        target_asin = record["target"]["parent_asin"]

        # Find target category
        category = asin_to_category.get(target_asin)

        if category is None:
            missing_category += 1
        else:
            category_counts[category] += 1

        # --------------------------------------------------
        # Remove rating + timestamp from target
        # --------------------------------------------------

        record["target"] = {
            "parent_asin": target_asin
        }

        fout.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )


In [4]:
print("\n==============================")
print("DATASET CATEGORY STATISTICS")
print("==============================")

print("Total records:", total_records)

for category, count in category_counts.items():
    percentage = (count / total_records) * 100
    print(f"{category}: {count} ({percentage:.2f}%)")

print("Missing category:", missing_category)

print("\nCleaned dataset saved at:")
print(CLEANED_FILE)


DATASET CATEGORY STATISTICS
Total records: 72909
Computers: 18059 (24.77%)
AMAZON FASHION: 2272 (3.12%)
Camera & Photo: 5139 (7.05%)
All Electronics: 24374 (33.43%)
Industrial & Scientific: 2527 (3.47%)
Sports & Outdoors: 255 (0.35%)
Amazon Devices: 3242 (4.45%)
Amazon Home: 804 (1.10%)
Cell Phones & Accessories: 6704 (9.20%)
Tools & Home Improvement: 1357 (1.86%)
Office Products: 915 (1.25%)
Apple Products: 840 (1.15%)
Home Audio & Theater: 3758 (5.15%)
Health & Personal Care: 167 (0.23%)
Car Electronics: 554 (0.76%)
Musical Instruments: 275 (0.38%)
Toys & Games: 55 (0.08%)
Portable Audio & Accessories: 40 (0.05%)
Automotive: 399 (0.55%)
All Beauty: 17 (0.02%)
GPS & Navigation: 137 (0.19%)
Amazon Fire TV: 20 (0.03%)
Arts, Crafts & Sewing: 11 (0.02%)
Pet Supplies: 12 (0.02%)
Video Games: 41 (0.06%)
Books: 9 (0.01%)
Software: 9 (0.01%)
Baby: 4 (0.01%)
Appliances: 5 (0.01%)
Buy a Kindle: 3 (0.00%)
Fire Phone: 1 (0.00%)
Missing category: 904

Cleaned dataset saved at:
../data2/output.jso

In [5]:
import json
from tqdm.notebook import tqdm

METADATA_FILE = "../data2/processed/meta_filtered.jsonl"
FINAL_METADATA_FILE =  "../data2/final_metadata.jsonl"

def preprocess_list(value, max_items=5):
    """
    Convert field to a clean list and keep at most max_items.
    """
    if value is None:
        return []

    if not isinstance(value, list):
        value = [value]

    cleaned = []

    for item in value:
        item = str(item).strip()

        if item:
            cleaned.append(item)

        if len(cleaned) == max_items:
            break

    return cleaned


total = 0

with open(METADATA_FILE, "r", encoding="utf-8") as fin, \
     open(FINAL_METADATA_FILE, "w", encoding="utf-8") as fout:

    for line in tqdm(
        fin,
        desc="Preprocessing metadata",
        colour="green"
    ):

        if not line.strip():
            continue

        product = json.loads(line)

       
        final_product = {
            "main_category": product.get("main_category", ""),
            "title": str(product.get("title", "")).strip(),
            
            "average_rating": product.get("average_rating"),
            "rating_number": product.get("rating_number"),

            "features": preprocess_list(
                product.get("features", []),
                max_items=5
            ),

            "description": preprocess_list(
                product.get("description", []),
                max_items=5
            ),

            "price": product.get("price"),

            "details": product.get(
                "details",
                {}
            ),

            "parent_asin": product.get("parent_asin", "")
        }

        fout.write(
            json.dumps(
                final_product,
                ensure_ascii=False
            ) + "\n"
        )

        total += 1


print("\n================================")
print("FINAL METADATA PREPROCESSING")
print("================================")

print("Products processed:", total)
print("Output file:", FINAL_METADATA_FILE)

Preprocessing metadata: 0it [00:00, ?it/s]


FINAL METADATA PREPROCESSING
Products processed: 577014
Output file: ../data2/final_metadata.jsonl
